In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
session.sql("USE DATABASE NHANES_DB").collect()
session.sql("USE SCHEMA RAW").collect()

df = session.table("NHANES_FINAL").to_pandas()

# Clean
df_clean = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP'])
diet_cols = ['SODIUM_MG','FIBER_G','SUGAR_G','CALORIES','PROTEIN_G','FAT_G']
df_clean[diet_cols] = df_clean[diet_cols].fillna(df_clean[diet_cols].median())
df_clean['BMI'] = df_clean['BMI'].fillna(df_clean['BMI'].median())

print("Ready! Shape:", df_clean.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd

# Features and target
features = ['SODIUM_MG', 'CALORIES', 'FIBER_G', 'BMI',
            'AGE', 'PROTEIN_G', 'FAT_G',
            'GENDER', 'ETHNICITY',
            'WEIGHT_KG', 'HEIGHT_CM']
target = 'HYPERTENSION_FLAG'

# Prepare data
ml_df = df_clean.dropna(subset=[target] + features)
X = ml_df[features]
y = ml_df[target]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,        
    min_samples_leaf=5,  
    max_features='sqrt', 
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train, y_train)

# Evaluate
rf_pred = rf_model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_pred))

# Feature importance
pd.Series(rf_model.feature_importances_, index=features).sort_values().plot(kind='barh')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

models = {
    'Random Forest': rf_model,
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results_rows   = []
confusion_rows = []
importance_rows = []

for name, model in models.items():

    if name != 'Random Forest':
        model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1] 

        # Performance metrics — unchanged
    results_rows.append({
        'MODEL_NAME': name,
        'ACCURACY':  round(accuracy_score(y_test,  preds), 4),
        'PRECISION': round(precision_score(y_test, preds), 4),
        'RECALL':    round(recall_score(y_test,    preds), 4),
        'F1_SCORE':  round(f1_score(y_test,        preds), 4),
        'AUC':       round(roc_auc_score(y_test,   probs), 4)
    })

    # Confusion matrix — simple version
    cm = confusion_matrix(y_test, preds)

    tn = cm[0][0]  # Actual No,  Predicted No  → Correct
    fp = cm[0][1]  # Actual No,  Predicted Yes → Wrong
    fn = cm[1][0]  # Actual Yes, Predicted No  → Wrong
    tp = cm[1][1]  # Actual Yes, Predicted Yes → Correct

    confusion_rows.append({'MODEL_NAME': name, 'ACTUAL': 'No Hypertension', 'PREDICTED': 'No Hypertension', 'COUNT': tn})
    confusion_rows.append({'MODEL_NAME': name, 'ACTUAL': 'No Hypertension', 'PREDICTED': 'Hypertension',    'COUNT': fp})
    confusion_rows.append({'MODEL_NAME': name, 'ACTUAL': 'Hypertension',    'PREDICTED': 'No Hypertension', 'COUNT': fn})
    confusion_rows.append({'MODEL_NAME': name, 'ACTUAL': 'Hypertension',    'PREDICTED': 'Hypertension',    'COUNT': tp})

        # Feature importance — unchanged
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        raw         = abs(model.coef_[0])
        importances = raw / raw.sum()

    for feat, imp in zip(features, importances):
        importance_rows.append({
            'MODEL_NAME': name,
            'FEATURE':    feat,
            'IMPORTANCE': round(float(imp), 4)
        })

results_df   = pd.DataFrame(results_rows)
confusion_df = pd.DataFrame(confusion_rows)
importance_df = pd.DataFrame(importance_rows)

print(results_df.to_string(index=False))

    

In [ ]:
# Check training vs test accuracy
train_accuracy = rf_model.score(X_train, y_train)
test_accuracy = rf_model.score(X_test, y_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:     {test_accuracy:.4f}")
print(f"Difference:        {train_accuracy - test_accuracy:.4f}")